# Simplest version — name, e-mail and room number in about 20 lines

Three regular expressions and the standard library only. It downloads the AIUB employee
feed, cuts it into one piece per faculty member, pulls the three fields, and writes
`output/simple_faculty.csv`.

The feed URL is the one the faculty page's own JavaScript loads (the site renders its cards
with JS, so the page HTML has no data). `AIUB_Faculty_Regex_Scraper.ipynb` shows how that URL
is discovered with regex and how the values get cleaned — this file is the bare version.

In [1]:
import re, csv, os, urllib.request

URL = ("https://www.aiub.edu/Files/Uploads/public-employee-profiles/"
       "employeeProfiles.json?v=1.0.11")
request = urllib.request.Request(URL, headers={"User-Agent": "Mozilla/5.0"})
text = urllib.request.urlopen(request, timeout=60).read().decode("utf-8", "replace")

# Regex 1: cut the text right before every employee object
RECORDS = re.compile(r'(?=\{"IsShowProfileDetails)')
records = [r for r in RECORDS.split(text) if r.startswith('{"IsShow')]

print("records downloaded:", len(records))

records downloaded: 446


In [2]:
# Regex 2-4: one pattern per field we need, plus the faculty filter
NAME  = re.compile(r'"Name":"([^"]*)"')
EMAIL = re.compile(r'"Email":"([^"]*)"')
ROOM  = re.compile(r'"RoomNo":"([^"]*)"')
FST   = re.compile(r'"Faculty":"FACULTY OF SCIENCE & TECHNOLOGY"')

def grab(pattern, record):
    match = pattern.search(record)
    return match.group(1) if match else ""

rows = [{"name": grab(NAME, r), "email": grab(EMAIL, r), "room": grab(ROOM, r)}
        for r in records if FST.search(r)]

print("science & technology faculty:", len(rows))
print()
for row in rows[:10]:
    print("{:<30} {:<26} {}".format(row["name"], row["email"], row["room"] or "-"))

science & technology faculty: 196

MD. ANWARUL KABIR              kabir@aiub.edu             DNO -216
MASHIOUR RAHMAN                mashiour@aiub.edu          DNGA05
SALMA BEGOM                    salma@aiub.edu             817
DR. MOHAMMED JASHIM UDDIN      drjashim@aiub.edu          1122
PROF. DR. KH. ABDUL MALEQUE    maleque@aiub.edu           DN0915
DR. SARFUDDIN AHMED TAREK      satarek@aiub.edu           DN0722
DR. AFROZA NAHAR               afroza@aiub.edu            DN0113
DR. S. A. M. MANZUR H. KHAN    manzur@aiub.edu            C
DR. FARZANA SABETH             fsabeth@aiub.edu           DN0721
DR. S. MOSADDEQ AHMED          smahmed@aiub.edu           DN0816


In [3]:
os.makedirs("output", exist_ok=True)
path = os.path.join("output", "simple_faculty.csv")

with open(path, "w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=["name", "email", "room"])
    writer.writeheader()
    writer.writerows(rows)

with open(path, encoding="utf-8") as handle:
    lines = len(handle.read().splitlines())

print("saved:", os.path.abspath(path))
print("csv lines:", lines, "(1 header +", lines - 1, "faculty)")

saved: /Users/roybipro/Developer/JNBook/NLP/Faculty-Data/output/simple_faculty.csv
csv lines: 197 (1 header + 196 faculty)


That is the whole simple version. Two things it deliberately does **not** do, which the full
notebook does:

* values stay exactly as the site stores them, so names are ALL CAPS (`MD. ANWARUL KABIR`) and
  room codes stay messy (`DNO -216`, `1122 (OR)`);
* the feed URL is written by hand instead of being discovered from the page with regex.